# A retrieval chain

**Retrieval** is the search that returns chunks for a question. A **retrieval chain** takes the user's question, retrieves the chunks, places those chunks and the question in one prompt, and sends that prompt to Groq.

The notes are the two accounts from lesson 46. The embedding is the same word list. The chat model is ChatGroq, from lesson 43.

Run the cells from top to bottom. Each Try it is one step.


## Retrieve the chunks

`as_retriever` turns the vector store into a search step. `search_kwargs={"k": 2}` asks for two chunks. `retriever.invoke` runs the search for the question. `format_docs` joins the chunk text into one string named `context`.

This cell stops after retrieval. It does not call Groq.


In [1]:
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore

WORDS = ("account", "open", "pending", "pen", "notebook", "stock", "price", "vin")

class WordEmbeddings(Embeddings):
    def embed_documents(self, texts):
        return [self.embed_query(text) for text in texts]

    def embed_query(self, text):
        cleaned = text.lower().replace(".", " ").replace(",", " ").replace("?", " ")
        found = cleaned.split()
        return [1.0 if word in found else 0.0 for word in WORDS]

def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

documents = [
    Document(page_content="Account 1001, VIN A100, status Open. Pen price is 1.5 and it is in stock.", metadata={"account": 1001}),
    Document(page_content="Account 1002, VIN B200, status Pending. Notebook price is 3.0 and it is not in stock.", metadata={"account": 1002}),
]
store = InMemoryVectorStore.from_documents(documents, WordEmbeddings())
retriever = store.as_retriever(search_kwargs={"k": 2})
question = "What is the price of the Pen?"
docs = retriever.invoke(question)
print(format_docs(docs))


Account 1001, VIN A100, status Open. Pen price is 1.5 and it is in stock.
Account 1002, VIN B200, status Pending. Notebook price is 3.0 and it is not in stock.


## Send the question and the chunks to Groq

The template has two blanks. `{context}` is the retrieved chunks. `{question}` is the user's question.

The `|` symbol connects steps, as in lesson 44. `prompt | model | parser` fills the prompt, calls Groq, and returns one string.

When `GROQ_API_KEY` is missing, the cell prints the prompt it would send, then a message, and it does not call the model. Lesson 38 shows how to set the key: in PowerShell, `$env:GROQ_API_KEY` is the name, and the text after `=` is your key. Start Jupyter from that same window.


In [ ]:
import os
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq

WORDS = ("account", "open", "pending", "pen", "notebook", "stock", "price", "vin")

class WordEmbeddings(Embeddings):
    def embed_documents(self, texts):
        return [self.embed_query(text) for text in texts]

    def embed_query(self, text):
        cleaned = text.lower().replace(".", " ").replace(",", " ").replace("?", " ")
        found = cleaned.split()
        return [1.0 if word in found else 0.0 for word in WORDS]

def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

documents = [
    Document(page_content="Account 1001, VIN A100, status Open. Pen price is 1.5 and it is in stock.", metadata={"account": 1001}),
    Document(page_content="Account 1002, VIN B200, status Pending. Notebook price is 3.0 and it is not in stock.", metadata={"account": 1002}),
]
store = InMemoryVectorStore.from_documents(documents, WordEmbeddings())
retriever = store.as_retriever(search_kwargs={"k": 2})
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer in one sentence from the notes only.\n\nNotes:\n{context}"),
    ("human", "{question}"),
])
question = "What is the price of the Pen?"
context = format_docs(retriever.invoke(question))
print(prompt.format(context=context, question=question))

key = os.environ.get("GROQ_API_KEY")
if not key:
    print("GROQ_API_KEY is missing. Set it in the environment, then run this cell again.")
else:
    model = ChatGroq(model="openai/gpt-oss-120b", api_key=key, temperature=0)
    chain = prompt | model | StrOutputParser()
    print(chain.invoke({"context": context, "question": question}))


## Interview question

**Question:** What does a retrieval chain send to the model?

**Spoken answer:** "It sends the user's question and the chunks the search returned. The vector store does the search. The prompt puts those chunks next to the question. Groq writes the answer from that prompt."


### Try it

`context` is an empty string, so the notes section is blank. Replace `context = ""` with `context = format_docs(docs)`. Before the change the printed notes are empty. After the change the notes include `1.5`.


In [3]:
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore

WORDS = ("account", "open", "pending", "pen", "notebook", "stock", "price", "vin")

class WordEmbeddings(Embeddings):
    def embed_documents(self, texts):
        return [self.embed_query(text) for text in texts]

    def embed_query(self, text):
        cleaned = text.lower().replace(".", " ").replace(",", " ").replace("?", " ")
        found = cleaned.split()
        return [1.0 if word in found else 0.0 for word in WORDS]

def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

documents = [
    Document(page_content="Account 1001, VIN A100, status Open. Pen price is 1.5 and it is in stock.", metadata={"account": 1001}),
    Document(page_content="Account 1002, VIN B200, status Pending. Notebook price is 3.0 and it is not in stock.", metadata={"account": 1002}),
]
store = InMemoryVectorStore.from_documents(documents, WordEmbeddings())
docs = store.similarity_search("What is the price of the Pen?", k=2)
context = ""
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer in one sentence from the notes only.\n\nNotes:\n{context}"),
    ("human", "{question}"),
])
print(prompt.format(context=context, question="What is the price of the Pen?"))


System: Answer in one sentence from the notes only.

Notes:

Human: What is the price of the Pen?


### Solution

Try the exercise above first. Unfold the next cell the same way as in lesson 1: click the blue bar on its left, then press Shift+Enter.


In [ ]:
# Solution
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore

WORDS = ("account", "open", "pending", "pen", "notebook", "stock", "price", "vin")

class WordEmbeddings(Embeddings):
    def embed_documents(self, texts):
        return [self.embed_query(text) for text in texts]

    def embed_query(self, text):
        cleaned = text.lower().replace(".", " ").replace(",", " ").replace("?", " ")
        found = cleaned.split()
        return [1.0 if word in found else 0.0 for word in WORDS]

def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

documents = [
    Document(page_content="Account 1001, VIN A100, status Open. Pen price is 1.5 and it is in stock.", metadata={"account": 1001}),
    Document(page_content="Account 1002, VIN B200, status Pending. Notebook price is 3.0 and it is not in stock.", metadata={"account": 1002}),
]
store = InMemoryVectorStore.from_documents(documents, WordEmbeddings())
docs = store.similarity_search("What is the price of the Pen?", k=2)
context = format_docs(docs)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer in one sentence from the notes only.\n\nNotes:\n{context}"),
    ("human", "{question}"),
])
print(prompt.format(context=context, question="What is the price of the Pen?"))


### Try it

The human message is the fixed text `Say hello.`, so the question never reaches the prompt. Replace `Say hello.` with `{question}`. Before the change the cell prints `Say hello.` After the change it prints `What is the price of the Pen?`


In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer in one sentence from the notes only.\n\nNotes:\n{context}"),
    ("human", "Say hello."),
])
text = prompt.format(context="Pen price is 1.5.", question="What is the price of the Pen?")
print(text)


System: Answer in one sentence from the notes only.

Notes:
Pen price is 1.5.
Human: Say hello.


### Solution

Try the exercise above first. Unfold the next cell the same way as in lesson 1: click the blue bar on its left, then press Shift+Enter.


In [ ]:
# Solution
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer in one sentence from the notes only.\n\nNotes:\n{context}"),
    ("human", "{question}"),
])
text = prompt.format(context="Pen price is 1.5.", question="What is the price of the Pen?")
print(text)
